# Text Categorization

In [3]:
import numpy as np
from collections import defaultdict, Counter
import math
import re

documents = [
    "The cat sat on the mat",
    "Dogs are loyal pets",
    "Cats and dogs can be friends",
    "He owns a domestic cat",
    "The car engine is powerful",
    "This vehicle is a red car",
    "He drives a fast sports car",
    "Cars require fuel to run"
]

labels = [
    "animal", "animal", "animal", "animal",   # first 4 docs = animals
    "vehicle", "vehicle", "vehicle", "vehicle" # last 4 docs = vehicles
]

In [4]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    return tokens

docs_tokens = [preprocess(doc) for doc in documents]

# Build vocabulary
vocab = sorted(list(set([w for doc in docs_tokens for w in doc])))



### 1. Naive Bayes (Multinomial)

In [5]:
def train_nb(docs_tokens, labels, vocab):
    classes = set(labels)
    class_doc_counts = Counter(labels)
    total_docs = len(labels)
    
    # Prior probabilities
    priors = {c: class_doc_counts[c]/total_docs for c in classes}
    
    # Term counts per class
    term_counts = {c: defaultdict(int) for c in classes}
    class_word_totals = {c:0 for c in classes}
    
    for doc, label in zip(docs_tokens, labels):
        for word in doc:
            term_counts[label][word] += 1
            class_word_totals[label] += 1
    
    # Likelihood with Laplace smoothing
    likelihood = {c:{} for c in classes}
    V = len(vocab)
    for c in classes:
        for word in vocab:
            likelihood[c][word] = (term_counts[c].get(word,0) + 1) / (class_word_totals[c] + V)
    
    return priors, likelihood

def predict_nb(doc_tokens, priors, likelihood, vocab):
    classes = priors.keys()
    scores = {}
    for c in classes:
        score = math.log(priors[c])
        for word in doc_tokens:
            if word in vocab:
                score += math.log(likelihood[c].get(word, 1/(len(vocab))))
        scores[c] = score
    return max(scores, key=scores.get)

priors, likelihood = train_nb(docs_tokens, labels, vocab)
preds_nb = [predict_nb(doc, priors, likelihood, vocab) for doc in docs_tokens]

print("Naive Bayes Predictions:", preds_nb)


Naive Bayes Predictions: ['animal', 'animal', 'animal', 'animal', 'vehicle', 'vehicle', 'vehicle', 'vehicle']


### 2. KNN (Cosine similarity)

In [6]:
def vectorize(doc_tokens, vocab):
    vec = np.zeros(len(vocab))
    counter = Counter(doc_tokens)
    for i, word in enumerate(vocab):
        vec[i] = counter[word]
    return vec

doc_vectors = np.array([vectorize(doc, vocab) for doc in docs_tokens])

def cosine_sim(vec1, vec2):
    if np.linalg.norm(vec1)==0 or np.linalg.norm(vec2)==0:
        return 0
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1)*np.linalg.norm(vec2))

def predict_knn(test_vec, train_vecs, train_labels, k=3):
    sims = [cosine_sim(test_vec, v) for v in train_vecs]
    topk_idx = np.argsort(sims)[-k:]
    top_labels = [train_labels[i] for i in topk_idx]
    counts = Counter(top_labels)
    return counts.most_common(1)[0][0]

preds_knn = [predict_knn(vec, doc_vectors, labels) for vec in doc_vectors]
print("KNN Predictions:", preds_knn)

KNN Predictions: ['animal', 'animal', 'animal', 'vehicle', 'vehicle', 'vehicle', 'vehicle', 'vehicle']


### 3. Rocchio Classifier

In [7]:
def train_rocchio(vectors, labels):
    classes = set(labels)
    centroids = {}
    for c in classes:
        idx = [i for i, l in enumerate(labels) if l==c]
        centroids[c] = np.mean(vectors[idx], axis=0)
    return centroids

def predict_rocchio(vec, centroids):
    sims = {c: cosine_sim(vec, centroids[c]) for c in centroids}
    return max(sims, key=sims.get)

centroids = train_rocchio(doc_vectors, labels)
preds_roc = [predict_rocchio(vec, centroids) for vec in doc_vectors]
print("Rocchio Predictions:", preds_roc)

Rocchio Predictions: ['animal', 'animal', 'animal', 'animal', 'vehicle', 'vehicle', 'vehicle', 'vehicle']


### 4. Decision Tree (simplified ID3 on words)

In [8]:
def entropy(labels):
    total = len(labels)
    counts = Counter(labels)
    ent = 0
    for count in counts.values():
        p = count/total
        ent -= p*math.log2(p)
    return ent

def information_gain(labels, word_present):
    total_entropy = entropy(labels)
    # Split
    yes_labels = [l for l, pres in zip(labels, word_present) if pres]
    no_labels = [l for l, pres in zip(labels, word_present) if not pres]
    yes_ratio = len(yes_labels)/len(labels)
    no_ratio = len(no_labels)/len(labels)
    gain = total_entropy - yes_ratio*entropy(yes_labels) - no_ratio*entropy(no_labels)
    return gain

def train_dt(docs_tokens, labels, vocab, depth=1):
    if len(set(labels))==1 or depth>3:
        return labels[0]
    # Find best split word
    best_word = None
    best_gain = -1
    for word in vocab:
        present = [word in doc for doc in docs_tokens]
        gain = information_gain(labels, present)
        if gain > best_gain:
            best_gain = gain
            best_word = word
    if best_word is None:
        return Counter(labels).most_common(1)[0][0]
    
    # Split
    present_idx = [i for i, doc in enumerate(docs_tokens) if best_word in doc]
    absent_idx = [i for i in range(len(docs_tokens)) if i not in present_idx]
    tree = {best_word:{}}
    tree[best_word]['yes'] = train_dt([docs_tokens[i] for i in present_idx],
                                      [labels[i] for i in present_idx], vocab, depth+1)
    tree[best_word]['no'] = train_dt([docs_tokens[i] for i in absent_idx],
                                     [labels[i] for i in absent_idx], vocab, depth+1)
    return tree

def predict_dt(tree, doc):
    if not isinstance(tree, dict):
        return tree
    word = list(tree.keys())[0]
    if word in doc:
        return predict_dt(tree[word]['yes'], doc)
    else:
        return predict_dt(tree[word]['no'], doc)

dtree = train_dt(docs_tokens, labels, vocab)
preds_dt = [predict_dt(dtree, doc) for doc in docs_tokens]
print("Decision Tree Predictions:", preds_dt)


Decision Tree Predictions: ['animal', 'animal', 'animal', 'animal', 'vehicle', 'vehicle', 'vehicle', 'vehicle']
